# Stability of an algorithm

In [ ]:
#    APM41012EP course notebook - Chapter 1 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Stability of an algorithm
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import plotly.graph_objects as go
import numpy as np
import plotly.io as pio
pio.templates.default = "seaborn"
import warnings
warnings.filterwarnings('ignore')

## Evaluation of a function (ill-conditioned case)

We study the conditioning of the function $P(x) = (x − 1)^6,$ first in a neighborhood of the value $x_0 = 1$. It is not hard to see that the condition number at $x$ is $\kappa(x) = x / \mathcal{P}(x) \sup_{|y-x|<\epsilon_M}\mathcal{P}^\prime(y)$ and is therefore approximately $6x/|x-1|$ in this neighborhood, which means that the problem is ill-conditioned there.

We propose to use three algorithms to evaluate the function on the interval $[0.995 , 1.005]$. These three algorithms are based on three mathematically equivalent forms and therefore have the same conditioning:
- the factorized form $P(x) = (1 − x)^6$
- the expanded form $P(x) = x^6 − 6x^5 + 15x^4 − 20x^3 + 15x^2 − 6x + 1$
- the form using Horner's algorithm $P(x) = ((((((x − 6)x + 15)x − 20)x + 15)x − 6)x + 1)$ 

In [ ]:
def P1(x):
    return (x-1)**6

def P2(x):
    return x**6-6*x**5+15*x**4-20*x**3+15*x**2-6*x+1

def P3(x):
    return ((((((x-6)*x+15)*x-20)*x+15)*x-6)*x+1)

To illustrate the behavior of this evaluation in the neighborhood of 1, the following cell plots the graphs of the function using the three algorithms to evaluate it.

In [ ]:
xmin = 0.995
xmax = 1.005
x = np.linspace(xmin,xmax, 1000)

vmin = min(np.min(P1(x)), np.min(P2(x)), np.min(P3(x)))
vmax = max(np.max(P1(x)), np.max(P2(x)), np.max(P3(x)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=P1(x), name='factorized form'))
fig.add_trace(go.Scatter(x=x, y=P2(x), name='developped form'))
fig.add_trace(go.Scatter(x=x, y=P3(x), name='Horner form'))
fig.update_yaxes(exponentformat='e', range=[1.1*vmin, 1.1*vmax])
fig.update_layout(xaxis_title="x", yaxis_title="P(x)", height=500, legend=dict(orientation="h", y=1.1))
fig.show()

We observe that the fact that the problem is ill-conditioned on this interval can have a significant impact on the errors depending on the algorithm used. The expanded and Horner forms give an error level almost two orders of magnitude larger than the factorized form. The oscillations have larger amplitude near 1, where the conditioning is poor, and gradually fade as one moves away from it, which seems to show that the evaluation errors are directly related to the conditioning.

The poor conditioning we have identified therefore implies potential evaluation difficulties and a strong sensitivity to the algorithm used. This means that the choice of algorithm will be all the more important as the problem is ill-conditioned, and we therefore need a tool to evaluate an algorithm. In conclusion, the conditioning gives important information intrinsic to the mathematical problem we wish to solve, but the algorithm used also plays a role and we need to be able to evaluate it.

## Evaluation of a function (well-conditioned case)

We consider the evaluation of the function $g(x)$ written in two different forms:

$$g_1(x) = \displaystyle \frac{1}{x(x+1)} \quad \text{and} \quad g_2(x) = \displaystyle \frac{1}{x} - \frac{1}{x+1}$$ 

In [ ]:
def g1(x):
    return 1/(x*(x+1))

def g2(x):
    return 1/x - 1/(x+1)

It is easy to see that in a neighborhood of the value of $x$ used, all the operations of algorithm $g_1$ are well-conditioned and the algorithm is numerically stable. On the other hand, the subtraction in a neighborhood where $1/x ≈ 1/(x + 1)$ is ill-conditioned and algorithm $g_2$ is numerically unstable with a potentially large constant that will generate a significant error.

### Evaluation of the function $g(x)$ for $x = 10000$

In [ ]:
from mpmath import mp

# Evaluation of the reference value using quadruple-precision floats
mp.prec = 113
gref = mp.mpf('1/100010000')
print(f"Reference value with {mp.dps} significant digits = {gref}")

# single precision
mp.prec = 24
print(f"\nMantissa size: {mp.prec} bits")
print(f"Reference value with {mp.dps} significant digits: {mp.nstr(gref, mp.dps, strip_zeros=False)}")
# Evaluation of the functions
x = mp.mpf('10000')
print(f"1st form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g1(x), mp.dps, strip_zeros=False)}")
print(f"2nd form with {mp.dps} significant digits: g(x={x}) = {g2(x)}")

# double precision
mp.prec = 53
print(f"\nMantissa size: {mp.prec} bits")
print(f"Reference value with {mp.dps} significant digits: {mp.nstr(gref, mp.dps, strip_zeros=False)}")
# Evaluation of the functions
x = mp.mpf('10000')
print(f"1st form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g1(x), mp.dps, strip_zeros=False)}")
print(f"2nd form with {mp.dps} significant digits: g(x={x}) = {g2(x)}")

After observing the reference value obtained with 33 significant digits, we use single precision on the one hand and double precision on the other. While algorithm 1 behaves very well with an error of the order of the round-off error, which indicates a very good backward stability constant for this well-conditioned problem, algorithm 2 leads us to a loss of 3 to 4 significant digits (recall that this problem is related to the value at which the function is evaluated), locally indicating an algorithm with a much less favorable backward stability constant, as expected from the conditioning of the subtraction of two very close numbers.

### Evaluation of the function $g(x)$ for $x = 100000000$

In [ ]:
# Evaluation of the reference value using quadruple-precision floats
mp.prec = 113
gref = mp.mpf('1/10000000100000000')
print(f"Reference value with {mp.dps} significant digits = {gref}")

# single precision
mp.prec = 24
print(f"\nMantissa size: {mp.prec} bits")
print(f"Reference value with {mp.dps} significant digits: {mp.nstr(gref, mp.dps, strip_zeros=False)}")
# Evaluation of the functions
x = mp.mpf('100000000')
print(f"1st form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g1(x), mp.dps, strip_zeros=False)}")
print(f"2nd form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g2(x), mp.dps, strip_zeros=False)}")

# double precision
mp.prec = 53
print(f"\nMantissa size: {mp.prec} bits")
print(f"Reference value with {mp.dps} significant digits: {mp.nstr(gref, mp.dps, strip_zeros=False)}")
# Evaluation of the functions
x = mp.mpf('100000000')
print(f"1st form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g1(x), mp.dps, strip_zeros=False)}")
print(f"2nd form with {mp.dps} significant digits: g(x={x}) = {mp.nstr(g2(x), mp.dps, strip_zeros=False)}")

While the loss of precision was annoying but not dramatic for the result at $x = 10000$, when we tackle this problem with algorithm 2 for a much larger $x$, we can face a catastrophic loss of significant digits: the loss of 6 to 7 significant digits in single precision is dangerous!, and even a significant degradation of the quality of the solution in double precision. Algorithm 1 remains unperturbed and continues to show excellent stability in both cases for this well-conditioned problem, leading to a forward error of the order of the machine precision.